In [27]:
df = pd.read_csv("sinh_vien_bo_hoc.csv")

In [28]:
!mkdir -p data
!mv sinh_vien_bo_hoc.csv gia_nha.csv data/

In [29]:
# -*- coding: utf-8 -*-
"""Bước 1: thấy accuracy đánh lừa trên dữ liệu lệch lớp.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
print("Kich thuoc bang:", df.shape)
print(df.head())
print()
print("So sinh vien moi lop:")
print(df["bo_hoc"].value_counts().to_string())
print(f"Ty le bo hoc: {df['bo_hoc'].mean():.3f}")
print()

X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
# stratify=y giữ đúng tỷ lệ bỏ học ở cả tập học lẫn tập kiểm tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print(f"Tap kiem tra: {len(y_te)} ban, {y_te.sum()} ban bo hoc")
print()

# Mô hình lười: ai cũng đoán là lớp đông nhất
luoi = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
du_luoi = luoi.predict(X_te)

lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
du_lr = lr.predict(X_te)

for ten, du in [("Mo hinh luoi", du_luoi), ("Hoi quy logistic", du_lr)]:
    acc = accuracy_score(y_te, du)
    tim_ra = ((du == 1) & (y_te == 1)).sum()
    print(f"{ten:17s} accuracy {acc:.4f}, "
          f"tim ra {tim_ra} tren {y_te.sum()} ban bo hoc")

Kich thuoc bang: (1200, 5)
   diem_tb  so_buoi_vang  tin_chi_no  gio_lam_them  bo_hoc
0      8.0             3           0             0       0
1      8.9             5           1             7       0
2      3.3             3           1             4       0
3      6.4             3           0             7       0
4      7.9             3           5            23       0

So sinh vien moi lop:
bo_hoc
0    1081
1     119
Ty le bo hoc: 0.099

Tap kiem tra: 300 ban, 30 ban bo hoc

Mo hinh luoi      accuracy 0.9000, tim ra 0 tren 30 ban bo hoc
Hoi quy logistic  accuracy 0.9467, tim ra 17 tren 30 ban bo hoc


In [31]:
"""Bước 2: tự đếm bốn ô của ma trận nhầm lẫn, rồi so với thư viện."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")   # hoặc "data/sinh_vien_bo_hoc.csv"
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
du = lr.fit(X_tr, y_tr).predict(X_te)
that = y_te.to_numpy()

# Tự đếm: mỗi ô là một cặp (nhãn thật, nhãn đoán)
tp = np.sum((that == 1) & (du == 1))
fp = np.sum((that == 0) & (du == 1))
fn = np.sum((that == 1) & (du == 0))
tn = np.sum((that == 0) & (du == 0))
print("Tu dem:")
print(f"  TP = {tp}  (doan bo hoc, dung la bo hoc)")
print(f"  FP = {fp}  (bao dong nham)")
print(f"  FN = {fn}  (bo sot)")
print(f"  TN = {tn}  (doan khong bo, dung la khong bo)")
print(f"  Tong = {tp + fp + fn + tn}")
print()

m = confusion_matrix(that, du)
print("confusion_matrix cua scikit-learn:")
print(m)
print("Thu tu khi trai ra: TN, FP, FN, TP =", m.ravel())
print()

acc = (tp + tn) / (tp + fp + fn + tn)
print(f"Accuracy tinh tu bon o: ({tp} + {tn}) / {len(that)} = {acc:.4f}")
print(f"So ban bi bao co nguy co: TP + FP = {tp + fp}")

Tu dem:
  TP = 17  (doan bo hoc, dung la bo hoc)
  FP = 3  (bao dong nham)
  FN = 13  (bo sot)
  TN = 267  (doan khong bo, dung la khong bo)
  Tong = 300

confusion_matrix cua scikit-learn:
[[267   3]
 [ 13  17]]
Thu tu khi trai ra: TN, FP, FN, TP = [267   3  13  17]

Accuracy tinh tu bon o: (17 + 267) / 300 = 0.9467
So ban bi bao co nguy co: TP + FP = 20


In [32]:
"""Bước 3: precision, recall, F1 tính tay và tính bằng thư viện."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")   # hoặc "data/sinh_vien_bo_hoc.csv"
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
du = lr.fit(X_tr, y_tr).predict(X_te)
tn, fp, fn, tp = confusion_matrix(y_te, du).ravel()

p = tp / (tp + fp)
r = tp / (tp + fn)
f1 = 2 * p * r / (p + r)
print("Tinh tay tu bon o:")
print(f"  Precision = {tp} / ({tp} + {fp}) = {p:.4f}")
print(f"  Recall    = {tp} / ({tp} + {fn}) = {r:.4f}")
print(f"  F1        = 2 x {p:.4f} x {r:.4f} / ({p:.4f} + {r:.4f})"
      f" = {f1:.4f}")
print()

print("Tinh bang scikit-learn:")
print(f"  Precision = {precision_score(y_te, du):.4f}")
print(f"  Recall    = {recall_score(y_te, du):.4f}")
print(f"  F1        = {f1_score(y_te, du):.4f}")
print()

print("Bao cao day du:")
print(classification_report(y_te, du, digits=4,
                            target_names=["khong bo", "bo hoc"]))

Tinh tay tu bon o:
  Precision = 17 / (17 + 3) = 0.8500
  Recall    = 17 / (17 + 13) = 0.5667
  F1        = 2 x 0.8500 x 0.5667 / (0.8500 + 0.5667) = 0.6800

Tinh bang scikit-learn:
  Precision = 0.8500
  Recall    = 0.5667
  F1        = 0.6800

Bao cao day du:
              precision    recall  f1-score   support

    khong bo     0.9536    0.9889    0.9709       270
      bo hoc     0.8500    0.5667    0.6800        30

    accuracy                         0.9467       300
   macro avg     0.9018    0.7778    0.8255       300
weighted avg     0.9432    0.9467    0.9418       300



In [33]:
"""Bước 4: đổi ngưỡng, và chọn ngưỡng đúng cách trên tập học."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")   # hoặc "data/sinh_vien_bo_hoc.csv"
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
xs = lr.predict_proba(X_te)[:, 1]   # xác suất bỏ học của từng bạn

print("Tap kiem tra, doi nguong")
print("nguong  so bao   TP   FP   FN  precision  recall      F1")
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    du = (xs >= t).astype(int)
    tp = int(((du == 1) & (y_te == 1)).sum())
    fp = int(((du == 1) & (y_te == 0)).sum())
    fn = int(((du == 0) & (y_te == 1)).sum())
    print(f"  {t:.1f}   {du.sum():5d}  {tp:3d}  {fp:3d}  {fn:3d}"
          f"    {precision_score(y_te, du):.4f}  "
          f"{recall_score(y_te, du):.4f}  {f1_score(y_te, du):.4f}")
print()

# Chọn ngưỡng đúng cách: chỉ dùng tập học, qua kiểm định chéo.
# Mỗi bạn trong tập học nhận xác suất từ một mô hình chưa từng thấy bạn ấy.
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5,
                           method="predict_proba")[:, 1]
cac_nguong = np.round(np.arange(0.05, 0.96, 0.05), 2)
f1_hoc = [f1_score(y_tr, (xs_hoc >= t).astype(int)) for t in cac_nguong]
t_tot = cac_nguong[int(np.argmax(f1_hoc))]
print(f"Nguong cho F1 cao nhat tren tap hoc: {t_tot}"
      f" (F1 kiem dinh cheo {max(f1_hoc):.4f})")

# Chọn xong mới dùng tập kiểm tra, và chỉ một lần
du = (xs >= t_tot).astype(int)
print(f"Tap kiem tra voi nguong {t_tot}:"
      f" precision {precision_score(y_te, du):.4f},"
      f" recall {recall_score(y_te, du):.4f},"
      f" F1 {f1_score(y_te, du):.4f}")

Tap kiem tra, doi nguong
nguong  so bao   TP   FP   FN  precision  recall      F1
  0.5      20   17    3   13    0.8500  0.5667  0.6800
  0.4      21   18    3   12    0.8571  0.6000  0.7059
  0.3      29   21    8    9    0.7241  0.7000  0.7119
  0.2      42   24   18    6    0.5714  0.8000  0.6667
  0.1      61   26   35    4    0.4262  0.8667  0.5714

Nguong cho F1 cao nhat tren tap hoc: 0.25 (F1 kiem dinh cheo 0.6473)
Tap kiem tra voi nguong 0.25: precision 0.6571, recall 0.7667, F1 0.7077


In [34]:
"""Bước 5: đường ROC, AUC, và so các mô hình đã học trong môn."""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("sinh_vien_bo_hoc.csv")   # hoặc "data/sinh_vien_bo_hoc.csv"
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
xs = lr.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]

# Mỗi ngưỡng cho một điểm của đường ROC
fpr, tpr, nguong = roc_curve(y_te, xs)
print("Duong ROC co", len(fpr), "diem. Vai diem dau:")
print("  bao dong nham   recall")
for a, b in list(zip(fpr, tpr))[:6]:
    print(f"  {a:13.4f}   {b:.4f}")
print(f"AUC cua logistic = {roc_auc_score(y_te, xs):.4f}")
print()

# So các mô hình đã học, cùng tập học, cùng tập kiểm tra
mo_hinh = {
    "Logistic": lr,
    "Naive Bayes": GaussianNB(),
    "KNN, k = 15": make_pipeline(StandardScaler(),
                                 KNeighborsClassifier(15)),
    "Cay sau 4": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Rung 200 cay": RandomForestClassifier(200, random_state=0),
    "SVM RBF": make_pipeline(StandardScaler(),
                             SVC(probability=True, random_state=0)),
}
print(f" {'Mo hinh':14s} {'AUC':>6s}  {'recall':>6s}  F1 (nguong 0.5)")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    p = mh.predict_proba(X_te)[:, 1]
    du = (p >= 0.5).astype(int)
    print(f" {ten:14s} {roc_auc_score(y_te, p):.4f}  "
          f"{recall_score(y_te, du):.4f}  {f1_score(y_te, du):.4f}")

Duong ROC co 28 diem. Vai diem dau:
  bao dong nham   recall
         0.0000   0.0000
         0.0000   0.0333
         0.0000   0.4000
         0.0074   0.4000
         0.0074   0.5333
         0.0111   0.5333
AUC cua logistic = 0.9449

 Mo hinh           AUC  recall  F1 (nguong 0.5)
 Logistic       0.9449  0.5667  0.6800
 Naive Bayes    0.9336  0.3667  0.5116
 KNN, k = 15    0.9274  0.2333  0.3684
 Cay sau 4      0.8803  0.4667  0.5957
 Rung 200 cay   0.9249  0.5667  0.6538
 SVM RBF        0.8996  0.5000  0.6250


In [35]:
"""Bước 6: kiểm định chéo với nhiều thước đo, và cân bằng lớp khi học."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")   # hoặc "data/sinh_vien_bo_hoc.csv"
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]

# Chia không phân tầng: số bạn bỏ học trong tập kiểm tra dao động
print("Khong stratify, so ban bo hoc trong tap kiem tra:")
dem = []
for rs in range(10):
    _, _, _, y_te_thu = train_test_split(X, y, test_size=0.25,
                                         random_state=rs)
    dem.append(int(y_te_thu.sum()))
print(" ", dem)
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

lr = make_pipeline(StandardScaler(), LogisticRegression())
print("Kiem dinh cheo 5 phan tren tap hoc:")
for tm in ["accuracy", "precision", "recall", "f1"]:
    d = cross_val_score(lr, X_tr, y_tr, cv=5, scoring=tm)
    print(f"  {tm:10s} {d.mean():.4f}")
print()

# Cân bằng lớp: phạt nặng hơn khi đoán sai lớp hiếm
cb = make_pipeline(StandardScaler(),
                   LogisticRegression(class_weight="balanced"))
print(f"Tap kiem tra  {'':12s} precision  recall      F1")
for ten, mh in [("Binh thuong", lr), ("Can bang lop", cb)]:
    du = mh.fit(X_tr, y_tr).predict(X_te)
    print(f"  {ten:12s}  {precision_score(y_te, du):.4f}"
          f"     {recall_score(y_te, du):.4f}  {f1_score(y_te, du):.4f}")


Khong stratify, so ban bo hoc trong tap kiem tra:
  [22, 26, 35, 29, 32, 27, 28, 20, 27, 23]

Kiem dinh cheo 5 phan tren tap hoc:
  accuracy   0.9289
  precision  0.7044
  recall     0.4810
  f1         0.5652

Tap kiem tra               precision  recall      F1
  Binh thuong   0.8500     0.5667  0.6800
  Can bang lop  0.4483     0.8667  0.5909


In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, roc_auc_score)
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
luoi = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

In [37]:
def danh_gia(y_that, y_doan):
    y_that = np.asarray(y_that)
    y_doan = np.asarray(y_doan)
    tp = np.sum((y_that == 1) & (y_doan == 1))
    fp = np.sum((y_that == 0) & (y_doan == 1))
    fn = np.sum((y_that == 1) & (y_doan == 0))
    tn = np.sum((y_that == 0) & (y_doan == 0))

    def chia(a, b):                 # chia an toan: mau bang 0 thi tra ve 0
        return a / b if b > 0 else 0.0

    acc = (tp + tn) / len(y_that)
    p = chia(tp, tp + fp)
    r = chia(tp, tp + fn)
    f1 = chia(2 * p * r, p + r)
    return acc, p, r, f1

for ten, mh in [("Luoi", luoi), ("Logistic", lr)]:
    du = mh.predict(X_te)
    print(ten, "tu viet :", np.round(danh_gia(y_te, du), 4))
    print(ten, "sklearn :", np.round([accuracy_score(y_te, du),
          precision_score(y_te, du, zero_division=0),
          recall_score(y_te, du), f1_score(y_te, du)], 4))

Luoi tu viet : [0.9 0.  0.  0. ]
Luoi sklearn : [0.9 0.  0.  0. ]
Logistic tu viet : [0.9467 0.85   0.5667 0.68  ]
Logistic sklearn : [0.9467 0.85   0.5667 0.68  ]


In [38]:
from sklearn.datasets import load_digits
from sklearn.metrics import classification_report

d = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    d.data, d.target, test_size=0.25, random_state=17, stratify=d.target)

svm = make_pipeline(StandardScaler(), SVC())
dud = svm.fit(Xd_tr, yd_tr).predict(Xd_te)
print(classification_report(yd_te, dud, digits=4))

bc = classification_report(yd_te, dud, output_dict=True)
cs = [str(i) for i in range(10)]
print("Recall thap nhat   :", min(cs, key=lambda c: bc[c]["recall"]))
print("Precision thap nhat:", min(cs, key=lambda c: bc[c]["precision"]))

              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        45
           1     0.9787    1.0000    0.9892        46
           2     1.0000    0.9773    0.9885        44
           3     0.9778    0.9565    0.9670        46
           4     0.9362    0.9778    0.9565        45
           5     1.0000    1.0000    1.0000        46
           6     0.9783    1.0000    0.9890        45
           7     0.9773    0.9556    0.9663        45
           8     0.9545    0.9767    0.9655        43
           9     0.9767    0.9333    0.9545        45

    accuracy                         0.9778       450
   macro avg     0.9779    0.9777    0.9777       450
weighted avg     0.9781    0.9778    0.9777       450

Recall thap nhat   : 9
Precision thap nhat: 4


In [39]:
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5,
                           method="predict_proba")[:, 1]
cac = np.round(np.arange(0.01, 0.99, 0.01), 2)
rec = np.array([recall_score(y_tr, (xs_hoc >= t).astype(int)) for t in cac])

hop_le = cac[rec >= 0.8]
t_chon = hop_le.max()          # nguong LON NHAT van dat recall >= 0.8
print("Nguong chon:", t_chon, "- recall kiem dinh cheo:", rec[cac == t_chon][0])

# Ap len tap kiem tra, mot lan duy nhat
xs = lr.predict_proba(X_te)[:, 1]
du = (xs >= t_chon).astype(int)
print("Recall    :", round(recall_score(y_te, du), 4))
print("Precision :", round(precision_score(y_te, du), 4))
print("So ban bi bao:", du.sum(), "tren", len(y_te))

Nguong chon: 0.17 - recall kiem dinh cheo: 0.8089887640449438
Recall    : 0.8
Precision : 0.5106
So ban bi bao: 47 tren 300


In [40]:
rf = RandomForestClassifier(200, random_state=0).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]
p_rf = rf.predict_proba(X_te)[:, 1]

fpr_l, tpr_l, _ = roc_curve(y_te, p_lr)
fpr_r, tpr_r, _ = roc_curve(y_te, p_rf)

plt.figure(figsize=(6, 6))
plt.plot(fpr_l, tpr_l, label=f"Logistic (AUC = {roc_auc_score(y_te, p_lr):.3f})")
plt.plot(fpr_r, tpr_r, label=f"Rung ngau nhien (AUC = {roc_auc_score(y_te, p_rf):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Doan bua")
plt.xlabel("Ty le bao dong nham (FPR)")
plt.ylabel("Recall (TPR)")
plt.legend()
plt.savefig("roc.png", dpi=150)
plt.show()

# Kiem tra hai duong co cat nhau khong
luoi_x = np.linspace(0, 1, 501)
hieu = np.interp(luoi_x, fpr_l, tpr_l) - np.interp(luoi_x, fpr_r, tpr_r)
doi_dau = np.where(np.diff(np.sign(hieu[hieu != 0])) != 0)[0]
print("So lan doi dau (cat nhau):", len(doi_dau))

So lan doi dau (cat nhau): 4


In [41]:
mo_hinh = {
    "Rung - thuong":    RandomForestClassifier(200, random_state=0),
    "Rung - can bang":  RandomForestClassifier(200, random_state=0,
                                               class_weight="balanced"),
    "SVM - thuong":     make_pipeline(StandardScaler(), SVC(random_state=0)),
    "SVM - can bang":   make_pipeline(StandardScaler(),
                                      SVC(class_weight="balanced", random_state=0)),
}
kq = {}
print(f"{'Mo hinh':16s} precision  recall      F1")
for ten, mh in mo_hinh.items():
    du = mh.fit(X_tr, y_tr).predict(X_te)
    kq[ten] = (precision_score(y_te, du, zero_division=0),
               recall_score(y_te, du), f1_score(y_te, du))
    print(f"{ten:16s} {kq[ten][0]:.4f}     {kq[ten][1]:.4f}  {kq[ten][2]:.4f}")

for m in ["Rung", "SVM"]:
    delta = np.array(kq[f"{m} - can bang"]) - np.array(kq[f"{m} - thuong"])
    print(m, "thay doi (precision, recall, F1):", np.round(delta, 4))

Mo hinh          precision  recall      F1
Rung - thuong    0.7727     0.5667  0.6538
Rung - can bang  0.7500     0.4000  0.5217
SVM - thuong     0.8824     0.5000  0.6383
SVM - can bang   0.4464     0.8333  0.5814
Rung thay doi (precision, recall, F1): [-0.0227 -0.1667 -0.1321]
SVM thay doi (precision, recall, F1): [-0.4359  0.3333 -0.0569]


In [42]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

nha = pd.read_csv("gia_nha.csv")
COT_GIA = "gia"                       # <- doi cho dung ten cot gia
Xn = nha.drop(columns=COT_GIA)
yn = nha[COT_GIA]
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.25, random_state=17)

du_n = LinearRegression().fit(Xn_tr, yn_tr).predict(Xn_te)

def ba_so(that, doan):
    return (mean_absolute_error(that, doan),
            np.sqrt(mean_squared_error(that, doan)),
            r2_score(that, doan))

truoc = np.array(ba_so(yn_te, du_n))

du_hong = du_n.copy()
du_hong[0] += 5          # +5 ty: neu gia tinh bang ty. Neu bang trieu thi cong 5000
sau = np.array(ba_so(yn_te, du_hong))

print("            MAE     RMSE      R2")
print("Truoc :", np.round(truoc, 4))
print("Sau   :", np.round(sau, 4))
print("Tang gap (lan):", np.round(sau / truoc, 2))

            MAE     RMSE      R2
Truoc : [0.2185 0.27   0.9759]
Sau   : [0.5519 1.4433 0.311 ]
Tang gap (lan): [2.53 5.35 0.32]
